In [16]:
from dotenv import load_dotenv
import os

from agents import Agent, Runner

In [17]:
load_dotenv(override=True)

True

In [18]:
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

In [19]:
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [20]:
first_agent = Agent(name="First Agent", instructions="You are a model to all general questions", model="gpt-4o-mini")

In [21]:
first_runner = await Runner.run(first_agent, "Who is the current US President")

In [22]:
print(first_runner.final_output)

As of my last knowledge update in October 2023, the President of the United States is Joe Biden. He has been in office since January 20, 2021. Please check the latest news for any updates or changes since then.


In [23]:
blog_writer = Agent(name="Blog Writer", instructions="You are to write a blog based off the topic I will send to you", model="gpt-4o-mini")

In [24]:
blog_runner = await Runner.run(blog_writer, "Write abt coding agent")

In [25]:
blog_runner.final_output

"**Unlocking the Future: The Rise of Coding Agents**\n\nIn today’s rapidly advancing technological landscape, the emergence of coding agents is transforming the way we approach software development. From automating repetitive tasks to enhancing our programming capabilities, coding agents are set to redefine what it means to be a developer. In this blog, we will explore what coding agents are, how they function, and the benefits they bring to the table.\n\n### What are Coding Agents?\n\nAt their core, coding agents are intelligent software programs designed to assist, augment, or even replace human coding tasks. These agents utilize artificial intelligence (AI), machine learning, and natural language processing to understand programming logic and make decisions based on user input. They can range from simple code auto-completion tools to advanced systems capable of generating entire applications with minimal input.\n\n### Types of Coding Agents\n\n1. **Code Assistants**: These agents pr

Function Tools

In [26]:
from agents import Agent, Runner, function_tool

In [ ]:
@function_tool
def get_weather(city: str) -> str:
  """
    Get the current weather for a city

    Args:
    city: The name of the city to get the current weather
  """
  weather_data = {
    "london": "Cloudy, 15C",
    "tokyo": "Sunny, 22C",
    "new york": "Rainy, 18C"
  }
  return weather_data.get(city.lower(), f"Weather data not available for this {city}")

In [28]:
weather_agent = Agent(
    name="WeatherBot",
    instructions="You help users check the weather, use the get_weather tool when asked about the weather",
    model="gpt-4o-mini",
    tools=[get_weather]
)

In [29]:
weather_runner = await Runner.run(weather_agent, "Whats the weather in Lagos")
print(weather_runner.final_output)

I couldn't find the weather data for Lagos. Could you please specify if it's Lagos in Nigeria or another location?


Sequential Tool Call

In [ ]:
@function_tool
def get_user(email: str)-> str:
    """ Look up the User ID from email """
    return "user_123"

@function_tool
def get_user_orders(user_id: str)-> str:
    """ Get orders for User ID """
    return "Order 1: Laptop, Order 2: Mouse"


@function_tool
def get_order_status(order_id: str)-> str:
    """ Get status of an order """
    return "Order 1: Delivery in Progress, Order 2: Awaiting Payment Confirmation"


order_agent = Agent(
    name="Order Assistant", 
    instructions="Help users check their order status, First use their eail to find UserID, then get the order and order status",
    model="gpt-4o-mini",
    tools=[get_user, get_user_orders, get_order_status]
 )

runner = await Runner.run(order_agent, "Whats the status of my order, my email is miracle@gmail.com")
print(runner.final_output)

Here is the status of your orders:

- **Order 1 (Laptop):** Delivery in Progress
- **Order 2 (Mouse):** Awaiting Payment Confirmation


Parallel Tool Calling

In [32]:
@function_tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"{city}: Sunny, 25°C"

@function_tool
def get_news(topic: str) -> str:
    """Get latest news on a topic."""
    return f"Latest {topic} news: ..."

@function_tool
def get_stock(symbol: str) -> str:
    """Get stock price."""
    return f"{symbol}: $150.00"


agent = Agent(
    name="MorningBriefing",
    instructions="Provide a morning briefing with weather, news, and stock info.",
    tools=[get_weather, get_news, get_stock]
)

runner = await Runner.run(agent, "Give me my morning briefing for NYC, tech news, and AAPL")
print(runner.final_output)

Morning briefing:

- Weather (NYC): Sunny, 25°C
- News (tech): Latest tech news news: ...
- Stock (AAPL): $150.00


Conditional Tool Calling

In [33]:
@function_tool
def search_web(query: str) -> str:
    """Search the web for current information."""
    return f"Web results for '{query}': ..."

@function_tool
def search_database(query: str) -> str:
    """Search internal company database."""
    return f"Database results for '{query}': ..."

@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents."""
    return f"Document results for '{query}': ..."

agent = Agent(
    name="SmartSearch",
    instructions="""You are a search assistant.
    
    Choose the right search based on the query:
    - For current events/general info → use search_web
    - For company/employee info → use search_database
    - For policy/procedure questions → use search_documents
    """,
    model="gpt-4o-mini",
    tools=[search_web, search_database, search_documents]
)

runner = await Runner.run(agent, "Find the latest company policy on remote work.")

print(runner.final_output)

I found the relevant document on the latest company policy regarding remote work. Please check it for detailed information. If you need anything specific from the document, let me know!


Multi-Agentic Systems


Building Multi-Agentic Systems with Handoff

In [ ]:
from agents import Agent, Runner, handoff, function_tool

@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents."""
    return f"Document results for '{query}': ..."

billing_agent = Agent(
    name="BillingSpecialist",
    instructions="""You handle billing questions:
    - Payment issues
    - Refunds
    - Invoice requests
    Be helpful and resolve issues quickly.""",
    tools=[search_documents]
)

technical_agent = Agent(
    name="TechnicalSupport",
    instructions="""You handle technical issues:
    - Bug reports
    - How-to questions
    - Feature requests
    Ask clarifying questions to understand the issue."""
)

triage_agent = Agent(
    name="Triage",
    instructions="""You are the first point of contact. Route customers immediately:
- Billing/payment/charge issues → transfer to BillingSpecialist RIGHT AWAY
- Technical problems → transfer to TechnicalSupport RIGHT AWAY

Do NOT ask clarifying questions if the intent is already clear. Transfer immediately.""",
    model="gpt-4o-mini",
    handoffs=[
        handoff(billing_agent),
        handoff(technical_agent)
    ]
)


result = await Runner.run(triage_agent, "I want to know how to fix my machine")
print(result.final_output)

Sure — I can help. What kind of machine is it, and what’s wrong with it?

Please share:
- the machine type/brand/model
- what it’s doing or not doing
- any error messages/lights/sounds
- when the problem started
- what you’ve already tried


Building Multi-Agentic Systems with Agent-as-a-tool

In [35]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"


from agents import Agent, Runner, function_tool

# Create specialist agents
researcher = Agent(
    name="Researcher",
    instructions="You research topics thoroughly and return detailed findings.",
    model="gpt-4o"
)

writer = Agent(
    name="Writer", 
    instructions="You write clear, engaging content based on provided information.",
    model="gpt-4o"
)

@function_tool
async def research_topic(topic: str) -> str:
    """
    Research a topic thoroughly.
    
    Args:
        topic: The topic to research
    """
    result = await Runner.run(researcher, f"Research this topic: {topic}")
    return result.final_output

@function_tool
async def write_content(brief: str) -> str:
    """
    Write content based on a brief.
    
    Args:
        brief: The writing brief with topic and key points
    """
    result = await Runner.run(writer, brief)
    return result.final_output

orchestrator = Agent(
    name="ContentManager",
    instructions="""You manage content creation.
    
    When asked to create content:
    1. Use research_topic to gather information
    2. Use write_content to create the final piece
    3. Review and present the result
    """,
    model="gpt-4o-mini",
    tools=[research_topic, write_content]
)

runner = await Runner.run(
    orchestrator,
    "Create a blog post about the benefits of meditation"
)

print(runner.final_output)

Here’s a blog post on the benefits of meditation:

---

**Unlock the Power Within: Exploring the Benefits of Meditation**

In today's fast-paced world, where chaos and stress seem like daily staples, meditation offers a sanctuary—a quiet retreat where inner peace and tranquility reign. Whether you're dealing with the ever-increasing responsibilities of work and family, or simply seeking a deeper connection with yourself, meditation stands as an ancient practice equipped to heal and rejuvenate both the mind and body. Let’s explore the myriad benefits of meditation and how this simple practice can lead to profound transformations in your life.

**1. Stress Reduction**

Arguably one of the most sought-after benefits of meditation is its powerful stress reduction capabilities. By engaging in mindfulness and focusing on the present moment, meditation helps the brain disconnect from the pressures and worries that accumulate over time. Studies have shown that regular meditation practice reduc